# Data Inspection
Loads a TWIX scan, extracts k-space, zero-fills it, and displays a preview as an animated GIF. This helps you determine the number of phase-encode lines collected and the offset. Then you can update those values in your configuration file accordingly.

### Loading packages and data

In [1]:
import yaml
import numpy as np
import utils.data_ingestion as di
import utils.gif as gif
from IPython.display import display

def load_config(config_file="config.yaml"):
    """
    Load configuration from a YAML file.

    Parameters
    ----------
    config_file : str
        Path to the YAML configuration file.

    Returns
    -------
    dict
        Parsed configuration data.
    """
    with open(config_file, "r") as f:
        config = yaml.safe_load(f)
    return config

# Read configuration
config = load_config()

# Extract paths and other parameters from config
twix_file = config["data"]["twix_file"]
dicom_folder = config["data"]["dicom_folder"]

# Read TWIX file (only the last scan by default in this example)
scans = di.read_twix_file(twix_file, include_scans=[-1], parse_pmu=False)

Software version: VD/VE (!?)

Scan  1


100%|██████████| 360M/360M [00:00<00:00, 1.34GB/s]

Read 1 scans from new_DATA/raw/meas_MID00056_FID00686_lax_Cine_rt_150meas_gre_75%FOV.dat


### Extracting k-space data, zero-filling, and displaying

In [8]:
# Extract raw k-space data
kspace = di.extract_image_data(scans[-1], full_kspace_shape=(150, 96, 20, 256), ref_scan=True)
# kspace = di.extract_image_data(scans[-1], ref_scan=True)
# kspace = kspace.reshape((150, -1, 20, 256))

# Display k-space as an animated GIF
preview = gif.display_kspace_as_gif(kspace, duration=0.2)
display(preview)

Extracted image data shape: (150, 96, 20, 256)


In [32]:
from utils.reconstruction import grappa_reconstruction, direct_ifft_reconstruction, tgrappa_reconstruction

# Figure out which lines are acquired
acquired_lines = np.where(~np.all(kspace == 0, axis=(0, 2, 3)))[0]
# Example: acquired_lines = [0, 2, 4, 6, 7, 8, 9, 10, 12, 14, 16]

# Figure out the largest contiguous block of acquired lines
contiguous_lines = np.split(acquired_lines, np.where(np.diff(acquired_lines) != 1)[0] + 1)
largest_block = max(contiguous_lines, key=len)
smallest_line, largest_line = largest_block[0], largest_block[-1]
print(f"Smallest line: {smallest_line}, Largest line: {largest_line}")

print("Performing GRAPPA...")
# images = grappa_reconstruction(kspace[:10, ...], calib_region=(smallest_line, largest_line), kernel_size=(3, 5))
images = tgrappa_reconstruction(kspace[:10, ...], calib_size=(20, 20), kernel_size=(5, 5))
# images = direct_ifft_reconstruction(kspace, use_conjugate_symmetry=True)
print("Reconstruction complete.")

Smallest line: 36, Largest line: 60
Performing GRAPPA...


Reconstruction complete.


In [33]:
# edited_images = np.rot90(images, k=1, axes=(1, 2))
# edited_images = np.flip(edited_images, axis=2)
# edited_images = edited_images[:, 64:-64, :]
# edited_images = np.rot90(edited_images, k=1, axes=(1, 2))
edited_images = np.flip(images, axis=1)
edited_images = edited_images[:, :, 64:-64]
preview = gif.display_images_as_gif(edited_images, notebook=True)
display(preview)

In [4]:
# # Extract raw k-space data
# kspace = di.extract_image_data(scans[-1])

# n_frames = di.get_num_frames(dicom_folder)
# n_coils = kspace.shape[1]

# # Reshape k-space into frames
# kspace = np.reshape(kspace, (n_frames, -1, n_coils, kspace.shape[2]))

# # Pull the total number of phase encodes and define offset
# extended_pe_lines = di.get_total_phase_encodes(dicom_folder)
# offset = 32  # Adjust if needed

# # Allocate zero-filled array
# kspace_zf = np.zeros((n_frames, extended_pe_lines, n_coils, kspace.shape[3]), dtype=np.complex64)
# kspace_zf[:, offset : offset + kspace.shape[1], :] = kspace

# # Display k-space as an animated GIF
# preview = gif.display_kspace_as_gif(kspace_zf, duration=0.2)
# display(preview)

In [6]:
# from utils.reconstruction import direct_ifft_reconstruction
# images = direct_ifft_reconstruction(kspace, extended_pe_lines, 0, use_conjugate_symmetry=False)
# images = np.rot90(images, k=1, axes=(1, 2))
# images = np.flip(images, axis=2)
# images = images[:, 64:-64, :]
# preview = gif.display_images_as_gif(images, notebook=True)
# display(preview)

In [6]:
for (i,mdb) in enumerate(scans[-1]['mdb']):
    if mdb.is_image_scan():
        print(
            mdb.cLin, # Specific line
            mdb.cRep, # Frame
            mdb.cSeg, # Segment
        )
    elif mdb.is_flag_set('PATREFSCAN'):
        print(
            mdb.cLin, # Specific line
            mdb.cRep, # Frame
            mdb.cSeg, # Segment
            "REFERENCE"
        )

0 0 0
2 0 1
4 0 2
6 0 3
8 0 4
10 0 5
12 0 6
14 0 7
16 0 8
18 0 9
20 0 10
22 0 11
24 0 12
26 0 13
28 0 14
30 0 15
32 0 16
34 0 17
36 0 18
37 0 19 REFERENCE
38 0 20
39 0 21 REFERENCE
40 0 22
41 0 23 REFERENCE
42 0 24
43 0 25 REFERENCE
44 0 26
45 0 27 REFERENCE
46 0 28
47 0 29 REFERENCE
48 0 30
49 0 31 REFERENCE
50 0 32
51 0 33 REFERENCE
52 0 34
53 0 35 REFERENCE
54 0 36
55 0 37 REFERENCE
56 0 38
57 0 39 REFERENCE
58 0 40
59 0 41 REFERENCE
60 0 42
62 0 43
64 0 44
66 0 45
68 0 46
70 0 47
72 0 48
74 0 49
76 0 50
78 0 51
80 0 52
82 0 53
84 0 54
86 0 55
88 0 56
90 0 57
92 0 58
94 0 59
0 1 0
2 1 1
4 1 2
6 1 3
8 1 4
10 1 5
12 1 6
14 1 7
16 1 8
18 1 9
20 1 10
22 1 11
24 1 12
26 1 13
28 1 14
30 1 15
32 1 16
34 1 17
36 1 18
37 1 19 REFERENCE
38 1 20
39 1 21 REFERENCE
40 1 22
41 1 23 REFERENCE
42 1 24
43 1 25 REFERENCE
44 1 26
45 1 27 REFERENCE
46 1 28
47 1 29 REFERENCE
48 1 30
49 1 31 REFERENCE
50 1 32
51 1 33 REFERENCE
52 1 34
53 1 35 REFERENCE
54 1 36
55 1 37 REFERENCE
56 1 38
57 1 39 REFERENCE
